In [10]:
# -- used librarys
import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

import seaborn as sns

from scipy import stats

from IPython.display import display

import pvlib
from pvlib.modelchain import ModelChain as Mc_Pva
from pvlib.location import Location 
from pvlib.pvsystem import PVSystem
from pvlib.solarposition import get_solarposition
from pvlib.temperature import TEMPERATURE_MODEL_PARAMETERS

import pyomo.environ as pyo
from pyomo.opt import SolverFactory

In [11]:
# -- Einlesen der Metadaten
# Norden
df_meta_north = pd.read_csv("./Nord 00591/Metadaten.txt", sep=";", skipinitialspace=True, skipfooter=1, engine="python", encoding="latin1")
height_station_north = float(df_meta_north.iloc[-1,4])
latlon_north = [float(df_meta_north.iloc[-1,3]), float(df_meta_north.iloc[-1,2])]
# Süden
df_meta_south = pd.read_csv("./Süd 05404/Metadaten.txt", sep=";", skipinitialspace=True, skipfooter=1, engine="python", encoding="latin1")
height_station_south = float(df_meta_south.iloc[-1,4]) 
latlon_south = [float(df_meta_south.iloc[-1,3]), float(df_meta_south.iloc[-1,2])]
# Zentrum
df_meta_center = pd.read_csv("./Zentrum 03231/Metadaten.txt", sep=";", skipinitialspace=True, skipfooter=1, engine="python", encoding="latin1")
height_station_center = float(df_meta_center.iloc[-1,4]) 
latlon_center = [float(df_meta_center.iloc[-1,3]), float(df_meta_center.iloc[-1,2])]

# list mit jeweiligen Metadaten für späteren Zugriff
list_latlons = [latlon_north, latlon_south, latlon_center]
list_heights_station = [height_station_north, height_station_south, height_station_center]

In [12]:
# -- Wetterdaten des Nordens einlesen und zusammenführen
# Importieren der Solardaten in J/(cm^2)
df_solar_north = pd.read_csv("./Nord 00591/solar.txt", sep=";", skipinitialspace=True, engine="python")
df_solar_north = df_solar_north.drop(columns=["STATIONS_ID", "QN", "SD_10", "LS_10", "eor"])
# Importieren der Lufttemperaturdaten in 2 m Höhe in °C
df_temperature_north = pd.read_csv("./Nord 00591/temperatur.txt", sep=";", skipinitialspace=True, engine="python")
df_temperature_north = df_temperature_north.drop(columns=["MESS_DATUM", "STATIONS_ID", "QN", "TM5_10", "RF_10", "TD_10", "eor"])
# Zusammenführen
df_north = df_solar_north.join(df_temperature_north, how="outer")

In [13]:
# -- Wetterdaten des Südens einlesen und zusammenführen
# Importieren der Solardaten in J/(cm^2)
df_solar_south = pd.read_csv("./Süd 05404/solar.txt", sep=";", skipinitialspace=True, engine="python")
df_solar_south = df_solar_south.drop(columns=["STATIONS_ID", "QN", "SD_10", "LS_10", "eor"])
# Importieren der Winddaten in m/s
df_wind_south = pd.read_csv("./Süd 05404/wind.txt", sep=";", skipinitialspace=True, engine="python")
df_wind_south = df_wind_south.drop(columns=["MESS_DATUM", "STATIONS_ID", "QN", "DD_10", "eor"])
# Importieren der Lufttemperaturdaten in 2 m Höhe in °C
df_temperature_south = pd.read_csv("./Süd 05404/temperatur.txt", sep=";", skipinitialspace=True, engine="python")
df_temperature_south = df_temperature_south.drop(columns=["MESS_DATUM", "STATIONS_ID", "QN", "TM5_10", "RF_10", "TD_10", "eor"])
# Zusammenführen
df_south = df_solar_south.join(df_temperature_south, how="outer")

In [14]:
# -- Wetterdaten des Zentrums einlesen und zusammenführen
# Importieren der Solardaten in J/(cm^2)
df_solar_center = pd.read_csv("./Zentrum 03231/solar.txt", sep=";", skipinitialspace=True, engine="python")
df_solar_center = df_solar_center.drop(columns=["STATIONS_ID", "QN", "SD_10", "LS_10", "eor"])
# Importieren der Lufttemperaturdaten in 2 m Höhe in °C
df_temperature_center = pd.read_csv("./Zentrum 03231/temperatur.txt", sep=";", skipinitialspace=True, engine="python")
df_temperature_center = df_temperature_center.drop(columns=["MESS_DATUM", "STATIONS_ID", "QN", "TM5_10", "RF_10", "TD_10", "eor"])
# Zusammenführen
df_center = df_solar_center.join(df_temperature_center, how="outer")

In [15]:
# -- Bereinigung der Daten - Ersetzen von Fehlerwert -999 durch Durchschnittswert zur gleichen Zeit der anderen Jahre
dict_error = {}
list_str_cardinal_direct = ["Norden", "Süden", "Zentrum"]
list_str_cardinal_direct_eng = ["north", "south", "center"]
indexdiff_one_year = 6 * 24 * 365 # Anzahl Indices pro Jahr - Schaltjahr vernachlässigt, da Verschiebung um 10 Minuten nur kleinen Einfluss
list_df_cardinal_direct = [df_north, df_south, df_center] # Bereinigung für jeden Datensatz
list_years = ["2020", "2021", "2022", "2023", "2024"] # Bereinigung für jedes Jahr, anhand der anderen Jahre
list_colnames = df_north.columns.astype(str)[1:] # north wird zwar genommen, jedoch gelten die Spaltennamen für alle Anlagen
for i_cd, df_cardinal_direct in enumerate(list_df_cardinal_direct):
    dict_error_measurement = {}
    for measurement in list_colnames: # Für alle gemessenen Werte (Solar -globale und diffuse Strahlung, Wind, Temperatur)
        counter_error_measurement = 0
        for index in range(df_cardinal_direct.shape[0]): # Für jeden Index (bzw. jeden Zeitstempel)
            if df_cardinal_direct.loc[index, measurement] == -999: # Alle Einträge, die Fehlerwert -999 haben
                counter_error_measurement += 1
                year_diff = 0 # Korrekturwert, der die Vergleichsjahre relativ zum betrachteten Jahr setzt
                for year in list_years: 
                    if str(df_cardinal_direct.iloc[index, 0])[:4] == year: # Filtern aufs jeweilige Jahr für spezielle Anwendung
                        list_compare_value = [] # Vergleichswerte zur gleichen Zeit aus den anderen Jahren werden hier eingefügt
                        list_manipulated_compare_factors = [(0-year_diff), (1-year_diff), (2-year_diff), (3-year_diff), (4-year_diff)]
                        for compare_factor in list_manipulated_compare_factors: # Zahlen in Liste stehen für järlichen Abstand, bsp. -1 => 1 Jahr Vergangenheit, 2 => 2 Jahre Zukunft
                            if df_cardinal_direct.loc[(index + indexdiff_one_year * compare_factor), measurement] != -999: # Filterung der Vergleichswerte mit Fehlerwert
                                list_compare_value.append(df_cardinal_direct.loc[(index + indexdiff_one_year * compare_factor), measurement])
                        df_cardinal_direct.loc[index, measurement] = np.mean(list_compare_value) # Fehlerwert durch Durchschnitt der Vergleichswerte ersetzen
                    year_diff += 1
        dict_error_measurement[measurement] = counter_error_measurement
    dict_error[list_str_cardinal_direct[i_cd]] = dict_error_measurement
    df_cardinal_direct["MESS_DATUM"] = pd.to_datetime(df_cardinal_direct["MESS_DATUM"], format="%Y%m%d%H%M") # Zeitstempel richtig einlesen
    df_cardinal_direct = df_cardinal_direct.rename(
        columns={"DS_10": "dhi", "GS_10": "ghi", "TT_10": "temp_air", "PP_10": "pressure"}
    )
    df_cardinal_direct = df_cardinal_direct.set_index("MESS_DATUM")
    df_cardinal_direct.index.name = None
    df_cardinal_direct["ghi"] *= (10000 / 600) # Umrechnungsfaktor von J/(cm² * 10min) in Ø W/(m²)
    df_cardinal_direct["dhi"] *= (10000 / 600) # Umrechnungsfaktor von J/(cm² * 10min) in Ø W/(m²)
    df_cardinal_direct["pressure"] *= 100 # Umrechnungsfaktor hPa in Pa    
    df_cardinal_direct["Ø globale Einstrahlung [kWh/(m²a)]"] = df_cardinal_direct["ghi"] * 8.76 # Umrechnungfaktor von W/(m²) in kWh/(m²a)
    list_df_cardinal_direct[i_cd] = df_cardinal_direct
df_error = pd.DataFrame(dict_error)
df_error = df_error.rename(index={
    "DS_10": "diffuse horizontale Einstrahlung",
    "GS_10": "globale horizontale Einstrahlung",
    "PP_10": "Luftdruck",
    "TT_10": "Temperatur in 2 m Höhe"
})
df_error.index.name = "Anzahl Fehlwerte"
display(df_error)

,Norden,Süden,Zentrum
Anzahl Fehlwerte,,,
diffuse horizontale Einstrahlung,604,6909,505
globale horizontale Einstrahlung,604,2447,505
Luftdruck,44,475,322
Temperatur in 2 m Höhe,64,588,0


In [16]:
# -- Gegenprüfung der Fehlwertbereinigung (Als Kommentar makiert, da Prozess nicht jedes Mal durchgelaufen werden muss)
for i_cd, df in enumerate(list_df_cardinal_direct):
    for index in range(len(df)):
        for measurement in range(5):
            if df.iloc[index, measurement] < -20:
                print(i_cd, index, measurement, df.iloc[index, measurement])

In [17]:
# -- 5-Jahresdurchschnitte der Wetterdaten aller Anlagen
dict_weather_mean = {}
list_weather_columns = []

for i_cd, df in enumerate(list_df_cardinal_direct):
    dict_colnames_weather_temp = {}
    list_weather_columns = [colname for colname in df.columns]
    for colname in df.columns:
        dict_colnames_weather_temp[colname] = float(round(df[colname].mean(), 2))
    dict_weather_mean[list_str_cardinal_direct[i_cd]] = dict_colnames_weather_temp
    
df_weather_mean = pd.DataFrame(data=dict_weather_mean)
df_weather_mean = df_weather_mean.rename(index={
    "dhi": "Ø diffuse Einstrahlung [W/m²]", 
    "ghi": "Ø globale Einstrahlung [W/m²]",
    "pressure": "Ø Luftdruck [Pa]",
    "Ø globale Einstrahlung [kWh/(m²a)]": "Ø globale Einstrahlungsenergie [kWh/(m²a)]",
	"temp_air": "Ø Lufttemperatur [°C]"
}).sort_index()

series_heights_station = pd.Series(list_heights_station, name="Anlagenhöhe [m ü. NHN]", index=df_weather_mean.columns)
df_weather_mean = pd.concat((df_weather_mean.T, series_heights_station), axis=1).T

df_weather_mean

,Norden,Süden,Zentrum
Ø Luftdruck [Pa],100946.36,96108.67,96332.04
Ø Lufttemperatur [°C],10.60,9.86,9.40
Ø diffuse Einstrahlung [W/m²],62.56,59.04,62.45
Ø globale Einstrahlung [W/m²],126.09,143.62,125.76
Ø globale Einstrahlungsenergie [kWh/(m²a)],1104.53,1258.10,1101.62
Anlagenhöhe [m ü. NHN],44.72,477.10,450.00


In [18]:
sandia_module = pvlib.pvsystem.retrieve_sam("SandiaMod")
sandia_module.loc["Nennleistung"] = sandia_module.loc["Impo"] * sandia_module.loc["Vmpo"]
sandia_module.loc["Nennleistung", :][(sandia_module.loc["Nennleistung", :] >= 250) & (sandia_module.loc["Nennleistung", :] <= 270)]

Schott_Solar_ASE_250_DGF_50__250___2007__E__        251.16
Schott_Solar_ASE_270_DGF_50__260___2007__E__        258.11
Schott_Solar_ASE_300_DGF_17__265___1999__E__        265.44
Schott_Solar_ASE_300_DGF_50__265___1999__E__         265.0
Suntech_STP260S_24_Vb__2007__E__                    260.05
Suntech_STP270S_24_Vb__2007__E__                    269.85
Suntech_STP270S_24_Vb_Module___2008_            265.472504
Suntech_STP270S_24_Vb_Module__2008__E__             269.85
Name: Nennleistung, dtype: object

In [19]:
# -- Modell der Berechnung der PVA-Erzeugungsdaten
dict_pva = {}
list_module_choice = ["Canadian_Solar_CS5P_220M___2009_", "Schott_Solar_ASE_270_DGF_50__260___2007__E__"] 
# system zur Berechnung erstellen
sandia_modules = pvlib.pvsystem.retrieve_sam("SandiaMod") # Datenbank für Solarmodule
cec_inverters = pvlib.pvsystem.retrieve_sam("CECInverter") # Datenbank für Wechselrichter
module = sandia_modules[list_module_choice[0]] # Referenzkonstellation 0: "Canadian_Solar_CS5P_220M___2009_"
inverter = cec_inverters["ABB__MICRO_0_25_I_OUTD_US_208__208V_"] 

temperature_parameters = TEMPERATURE_MODEL_PARAMETERS["sapm"]["open_rack_glass_glass"]

all_min_temps = [df["temp_air"].min() for df in list_df_cardinal_direct]
min_temperature = min(all_min_temps) # -18.1
# Modul- und Wechselrichterkonfiguration
def modulecount():
    max_dc_power_inverter = inverter["Pdco"] # max. zulässige PV-Leistung
    max_power_module = module["Impo"] * module["Vmpo"]
    max_modulecount = int(max_dc_power_inverter / max_power_module) # Maximale Modulanzahl pro Inverter
    V_oc_min_module = module["Voco"] * (1 + (module["Bvoco"] / 100) * (25 - min_temperature)) # Leerlaufspannung bei niedrigster Temperatur
    I_sc_module = module["Isco"]
    max_modules_per_string = int(inverter["Vdcmax"] / V_oc_min_module) # Maximale Modulanzahl pro String, max. WR-Spannung nicht überschreiten
    max_strings_per_inverter = int(inverter["Idcmax"] / I_sc_module)
    dict_modulecount = {
        "max_power_module": max_power_module,
        "max_module_per_string": max_modules_per_string,
        "max_strings_per_inverter": max_strings_per_inverter,
        "max_modules_per_inverter": max_modulecount,
        "max_dc_power_inverter": max_dc_power_inverter
    } # veraltet: {'strings': 3, 'modules_per_string': 8, 'modules_per_inverter': 28, 'dc_power': 5271.761498976}
    return dict_modulecount
moduleconfig = modulecount()
print(moduleconfig) # sollte für Auslegung verwendet werden, jedoch nicht optimal, wird nur noch für Informationen verwendet
modules_per_string = 1
strings_per_inverter = 1
max_dc_power = moduleconfig["max_power_module"] * modules_per_string * strings_per_inverter

system_pva = PVSystem(
    surface_tilt=30, 
    surface_azimuth=180, 
    module_parameters=module, 
    inverter_parameters=inverter, 
    temperature_model_parameters=temperature_parameters,
    modules_per_string=modules_per_string,
    strings_per_inverter=strings_per_inverter
)
# Jeder Anlage anhand ihrer Metadaten eine Location und eine ModelChain zuordnen, mit der die Erzeugungsdaten berechnet werden
list_locations = []
for i_cd, df in enumerate(list_df_cardinal_direct): # i_cd ist ein Index, wobei 0=Norden, 1=Osten, ..., 4=Zentrum
    latitude = list_latlons[i_cd][0]
    longitude = list_latlons[i_cd][1]
    altitude = list_heights_station[i_cd]
    location = Location(
        latitude=latitude, 
        longitude=longitude,
        altitude=altitude,
        tz="Europe/Berlin"
    )
    list_locations.append(location)

    modelchain_pva = Mc_Pva(
        system=system_pva, 
        location=location,
        aoi_model="sapm",
        spectral_model="sapm",
        losses_model="pvwatts"
    )
    print(modelchain_pva.system.pvwatts_losses())
    # Direktstrahlung (dni) anhand von Globalstrahlung (ghi), Diffusstrahlung (dhi) und Sonnenzenith berechnen
    solpos = get_solarposition(
        time=df.index, 
        latitude=latitude, 
        longitude=longitude,
        altitude=altitude
        ) # Sonnenposition berechnen
    dni = pvlib.irradiance.dirint(
        ghi=df["ghi"],
        solar_zenith=solpos["zenith"],
        times=df.index,
        pressure=df["pressure"],
    ) # ghi = dhi + dni * cos(theta)
    df["dni"] = dni.clip(lower=0)
    # ModelChains durchführen und results speichern
    modelchain_pva.run_model(df) # ModelChain für jeweiliges Dataframe erstellen - Nutzt Spalten temp_air, ghi, dni und dhi
    scaling_factor_pva = 1_000_000 / max_dc_power

    ac = modelchain_pva.results.ac.copy()
    ac = ac.clip(lower=0).fillna(0)

    P_pva_scalable = ac * scaling_factor_pva # auf 1.000.000 W (1 MW) skaliert
    list_df_cardinal_direct[i_cd]["P_pva_scalable"] = P_pva_scalable # Ergebnisse in dict abspeichern

    # Nur für Visualisierung
    ac = ac # .loc["2020-05-18 12:20:00":"2020-05-18 14:20:00"]
    dc = modelchain_pva.results.dc.p_mp.clip(lower=0).fillna(0) # .loc["2020-05-18 12:20:00":"2020-05-18 14:20:00"]

    dc_power = (dc * scaling_factor_pva).mean() 
    inverter_efficiency_weighted_percent = (ac.mean() / dc.mean()) * 100
    inverter_efficiency_unweighted_percent = (ac / dc).mean() * 100
    effective_irradiance = modelchain_pva.results.total_irrad["poa_global"].fillna(0).mean()

    dict_pva_temp = {
        "Ø DC-Leistung [kW]": (dc_power / 1_000), 
        "Ø AC-Leistung [kW]": (P_pva_scalable.mean() / 1_000),
        "Ø WR-Wirkungsgrad (gewichtet) [%]": inverter_efficiency_weighted_percent, 
        "Ø WR-Wirkungsgrad (ungewichtet) [%]": inverter_efficiency_unweighted_percent, 
        "Ø Globalstrahlung Modulebene [W/m²]": effective_irradiance,
        "Ø Volllaststunden [h/a]": ((P_pva_scalable.mean() / 1_000_000 ) * 8760)
    }
    dict_pva[list_str_cardinal_direct[i_cd]] = dict_pva_temp

df_pva = pd.DataFrame(dict_pva)
display(df_pva.round(2))

{'max_power_module': 219.656729124, 'max_module_per_string': 0, 'max_strings_per_inverter': 1, 'max_modules_per_inverter': 1, 'max_dc_power_inverter': 259.588593}
14.075660688264469
14.075660688264469
14.075660688264469


,Norden,Süden,Zentrum
Ø DC-Leistung [kW],119.26,127.78,113.74
Ø AC-Leistung [kW],112.27,120.49,107.00
Ø WR-Wirkungsgrad (gewichtet) [%],94.13,94.29,94.07
Ø WR-Wirkungsgrad (ungewichtet) [%],79.15,81.14,78.59
Ø Globalstrahlung Modulebene [W/m²],152.99,163.84,145.00
Ø Volllaststunden [h/a],983.46,1055.46,937.31
